# 02 — Pareto Testing for Multi-Objective Risk Control

This notebook demonstrates a simplified version of **Pareto Testing**:

- We consider **two objectives** per hyperparameter configuration:
  1. A **risk** to be controlled (e.g., accuracy reduction).
  2. A **cost** to be minimized (e.g., computational cost / latency).
- We:
  1. Generate a synthetic CSV `../data/sample_PT_losses.csv` with multiple samples
     per hyperparameter for both risk and cost.
  2. Split samples into an **optimization split** and a **testing split**.
  3. Use the optimization split to:
     - Estimate mean risk and mean cost for each configuration.
     - Recover the **Pareto frontier** in (risk, cost) space.
     - Order configurations on the frontier by risk / p-value.
  4. Use the testing split to run a simple **Fixed-Sequence Testing (FST)** over
     the Pareto frontier and select risk-controlling configurations.
  5. Among the accepted configurations, we pick those with low cost.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

PT_CSV = DATA_DIR / "sample_PT_losses.csv"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("PT CSV path:", PT_CSV)


In [ ]:
def generate_sample_pt_csv(path: Path, num_lambdas: int = 40, num_samples: int = 120, seed: int = 1):
    """Generate a synthetic CSV for Pareto Testing.

    Format:
    - One row per hyperparameter configuration λ.
    - Columns:
        - 'lambda_id' (int)
        - 'risk_1..risk_M'  (risk to be controlled, smaller is better)
        - 'cost_1..cost_M'  (cost, smaller is better)
    """
    rng = np.random.default_rng(seed)
    lambda_ids = np.arange(num_lambdas)

    # Construct a trade-off: configs with lower risk tend to have higher cost, and vice versa
    base_risks = np.linspace(0.02, 0.30, num_lambdas)   # mean risk per λ
    base_costs = np.linspace(0.9, 0.2, num_lambdas)     # mean cost per λ (e.g., normalized FLOPs)

    risk_samples = []
    cost_samples = []
    for mu_r, mu_c in zip(base_risks, base_costs):
        r = np.clip(rng.normal(loc=mu_r, scale=0.03, size=num_samples), 0.0, 1.0)
        c = np.clip(rng.normal(loc=mu_c, scale=0.05, size=num_samples), 0.0, 1.5)
        risk_samples.append(r)
        cost_samples.append(c)

    risk_arr = np.stack(risk_samples, axis=0)  # (num_lambdas, num_samples)
    cost_arr = np.stack(cost_samples, axis=0)

    cols = ["lambda_id"]
    risk_cols = [f"risk_{i+1}" for i in range(num_samples)]
    cost_cols = [f"cost_{i+1}" for i in range(num_samples)]
    cols_all = cols + risk_cols + cost_cols

    data = np.column_stack([lambda_ids, risk_arr, cost_arr])
    df = pd.DataFrame(data, columns=cols_all)
    df["lambda_id"] = df["lambda_id"].astype(int)

    df.to_csv(path, index=False)
    return df


if not PT_CSV.exists():
    print("CSV does not exist — generating synthetic sample_PT_losses.csv ...")
    df_pt = generate_sample_pt_csv(PT_CSV)
else:
    print("CSV already exists — loading:", PT_CSV)
    df_pt = pd.read_csv(PT_CSV)

print("\nFirst few rows of the PT CSV:")
display(df_pt.head())

print("\nColumns:")
print(df_pt.columns.tolist())


In [ ]:
lambda_ids = df_pt["lambda_id"].values
risk_cols = [c for c in df_pt.columns if c.startswith("risk_")]
cost_cols = [c for c in df_pt.columns if c.startswith("cost_")]

risk_mat = df_pt[risk_cols].values  # shape (L, M)
cost_mat = df_pt[cost_cols].values

num_lambdas, num_samples = risk_mat.shape
print(f"Number of hyperparameters: {num_lambdas}")
print(f"Number of samples per objective: {num_samples}")

# Split samples into optimization and testing
m1 = num_samples // 2
m2 = num_samples - m1

risk_opt = risk_mat[:, :m1]
risk_test = risk_mat[:, m1:]

cost_opt = cost_mat[:, :m1]
cost_test = cost_mat[:, m1:]

Q1_opt = risk_opt.mean(axis=1)   # controlled risk
Q1_test = risk_test.mean(axis=1)
Q2_opt = cost_opt.mean(axis=1)   # free objective (cost)
Q2_test = cost_test.mean(axis=1)

plt.figure()
plt.scatter(Q1_opt, Q2_opt, alpha=0.7)
plt.xlabel("Mean risk (Q1_opt)")
plt.ylabel("Mean cost (Q2_opt)")
plt.title("Hyperparameters in (risk, cost) space — optimization split")
plt.grid(True)
plt.show()


In [ ]:
def pareto_front_indices(risks, costs):
    """Return indices of Pareto-optimal configurations (minimization in both dims)."""
    n = len(risks)
    is_dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        if is_dominated[i]:
            continue
        for j in range(n):
            if i == j:
                continue
            if (risks[j] <= risks[i]) and (costs[j] <= costs[i]) and (
                (risks[j] < risks[i]) or (costs[j] < costs[i])
            ):
                is_dominated[i] = True
                break
    return np.where(~is_dominated)[0]


pareto_idx = pareto_front_indices(Q1_opt, Q2_opt)
print(f"Pareto frontier size: {len(pareto_idx)} out of {num_lambdas}")

plt.figure()
plt.scatter(Q1_opt, Q2_opt, alpha=0.2, label="All configs")
plt.scatter(Q1_opt[pareto_idx], Q2_opt[pareto_idx], c="red", label="Pareto frontier")
plt.xlabel("Mean risk (Q1_opt)")
plt.ylabel("Mean cost (Q2_opt)")
plt.title("Pareto frontier (optimization split)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
def hoeffding_p_value(r_hat, alpha, n):
    if r_hat >= alpha:
        return 1.0
    return float(np.exp(-2.0 * n * (alpha - r_hat) ** 2))


alpha = 0.10  # risk level to control (e.g., max acceptable accuracy reduction)
delta = 0.10  # FWER tolerance

# Compute p-values on the testing split for each λ (will be used in FST)
p_values_test = np.array([hoeffding_p_value(r, alpha, m2) for r in Q1_test])

# Restrict to Pareto frontier
pf_risks = Q1_opt[pareto_idx]
pf_costs = Q2_opt[pareto_idx]
pf_ids = lambda_ids[pareto_idx]

# Order Pareto frontier by increasing risk (equivalently, decreasing expected p-value)
order_pf = np.argsort(pf_risks)
pf_ordered_idx = pareto_idx[order_pf]

print("Ordered Pareto frontier (by Q1_opt increasing):")
print(list(zip(lambda_ids[pf_ordered_idx], Q1_opt[pf_ordered_idx], Q2_opt[pf_ordered_idx]))[:10])


In [ ]:
# Fixed-Sequence Testing (FST) along the Pareto frontier

accepted_indices = []
for idx in pf_ordered_idx:
    p = p_values_test[idx]
    if p < delta:
        accepted_indices.append(idx)
    else:
        break  # stop at first non-rejection

accepted_indices = np.array(accepted_indices, dtype=int)
accepted_lambdas = lambda_ids[accepted_indices]

print(f"Alpha (risk level): {alpha}")
print(f"Delta (FWER target): {delta}")
print(f"Number of accepted configs on Pareto frontier: {len(accepted_indices)}")
print("Accepted lambda_ids:", accepted_lambdas.tolist())

if len(accepted_indices) > 0:
    # Among accepted, pick the one with minimal cost (testing split cost)
    best_idx = accepted_indices[np.argmin(Q2_test[accepted_indices])]
    best_lambda = lambda_ids[best_idx]
    print(f"\nBest config among accepted (minimizing Q2_test): lambda_id={best_lambda}, "
          f"risk_test={Q1_test[best_idx]:.4f}, cost_test={Q2_test[best_idx]:.4f}")


In [ ]:
# Visualize accepted configurations on the optimization split

plt.figure()
plt.scatter(Q1_opt, Q2_opt, alpha=0.15, label="All configs")
plt.scatter(Q1_opt[pareto_idx], Q2_opt[pareto_idx], c="red", label="Pareto frontier")
if len(accepted_indices) > 0:
    plt.scatter(Q1_opt[accepted_indices], Q2_opt[accepted_indices],
                edgecolors="black", facecolors="none", s=100, label="Accepted (FST)")
plt.xlabel("Mean risk (Q1_opt)")
plt.ylabel("Mean cost (Q2_opt)")
plt.title("Pareto Testing: accepted configs (optimization split view)")
plt.legend()
plt.grid(True)
plt.show()
